# Building SQL in Lua

An honest notebook. The Diluvium WASM artifact does not link SQLite —
nothing in the amalgamation compiles it in — so there is no `sqlite`
library to call **from a cell**, and this page will not pretend otherwise.

> **Looking for the database?** It is in **SQLite, through a hostcall** —
> real SQLite, on the other side of the sandbox boundary. It is not a Lua
> library: a program granted `host:sql/query` asks its host for rows the
> way it asks for the time. Nothing about *this* notebook changes, because
> a sealed program still has no database inside it — which is the whole
> point of a capability. This one is about what a program does when it has
> no host to ask, and the last section here shows the same query put the
> other way for comparison.

What it does instead is the next real thing, and arguably the more
instructive one: build a small relational engine in Lua — filter,
project, sort, group, join — then put a **working SQL parser** on top of
it, so that by the end this runs, for real, in this kernel:

```sql
SELECT item, amount FROM orders WHERE amount >= 90 ORDER BY amount DESC
```

Along the way it exercises the parts of Diluvium a database binding
would lean on anyway: closures, `switch`, f-strings, `pcall`, and
string patterns. The last section says exactly what real SQLite in the
artifact would take.


In [ ]:
db = {
  customers = {
    { id = 1, name = "ada",     city = "London"    },
    { id = 2, name = "grace",   city = "Arlington" },
    { id = 3, name = "edsger",  city = "Austin"    },
    { id = 4, name = "annie",   city = "London"    },
    { id = 5, name = "alan",    city = "Wilmslow"  },
    { id = 6, name = "barbara", city = "Cambridge" },
  },
  orders = {
    { id = 101, customer_id = 1, item = "valves",      amount = 120 },
    { id = 102, customer_id = 1, item = "punch cards", amount = 45  },
    { id = 103, customer_id = 2, item = "relays",      amount = 260 },
    { id = 104, customer_id = 3, item = "chalk",       amount = 12  },
    { id = 105, customer_id = 4, item = "valves",      amount = 80  },
    { id = 106, customer_id = 4, item = "tape",        amount = 33  },
    { id = 107, customer_id = 5, item = "tape",        amount = 90  },
    { id = 108, customer_id = 2, item = "punch cards", amount = 150 },
    { id = 109, customer_id = 6, item = "chalk",       amount = 18  },
    { id = 110, customer_id = 5, item = "valves",      amount = 210 },
  },
}

for name, t in pairs(db) do print($"{name}: {#t} rows") end


## Relational verbs are ten-line functions

A query engine's core is four verbs: filter rows (`WHERE`), project
columns (`SELECT`), sort (`ORDER BY`), and truncate (`LIMIT`). Over
Lua tables each is a handful of lines, and composing them *is* running a
query — the SQL later is only a nicer spelling.

(`select` is already a Lua stdlib function, so the projection verb is
`select_cols` — the first of many small collisions a real binding has
to care about.)


In [ ]:
-- Every verb takes a list of rows and returns a new one; nothing mutates.

function where(rows, pred)
  local out = {}
  for _, r in ipairs(rows) do if pred(r) then out[#out + 1] = r end end
  return out
end

function select_cols(rows, cols)
  local out = {}
  for i, r in ipairs(rows) do
    local slim = {}
    for _, c in ipairs(cols) do slim[c] = r[c] end
    out[i] = slim
  end
  return out
end

function order_by(rows, col, desc)
  local out = { table.unpack(rows) }
  table.sort(out, function(a, b)
    if desc then return a[col] > b[col] end
    return a[col] < b[col]
  end)
  return out
end

function limit(rows, n)
  local out = {}
  for i = 1, math.min(n, #rows) do out[i] = rows[i] end
  return out
end

-- And one to look at results with.
function show(rows, cols)
  if #rows == 0 then print("(no rows)") return end
  if not cols then
    cols = {}
    for k in pairs(rows[1]) do cols[#cols + 1] = k end
    table.sort(cols)
  end
  print(table.concat(cols, " | "))
  for _, r in ipairs(rows) do
    local vals = {}
    for i, c in ipairs(cols) do vals[i] = tostring(r[c]) end
    print(table.concat(vals, " | "))
  end
end

-- the three biggest orders
show(limit(order_by(db.orders, "amount", true), 3), { "id", "item", "amount" })


`GROUP BY` is one more verb and an aggregate or two. One detail is
doing quiet work: the groups keep **first-appearance order** in a side
list, because iterating a Lua table with `pairs` promises no order at
all — the same reason a real database makes you say `ORDER BY` if you
mean it.


In [ ]:
function group_by(rows, col)
  local groups, seen = {}, {}
  for _, r in ipairs(rows) do
    local k = r[col]
    if not groups[k] then groups[k] = {}; seen[#seen + 1] = k end
    table.insert(groups[k], r)
  end
  return groups, seen   -- seen keeps first-appearance order; pairs() would not
end

function sum(rows, col)
  local total = 0
  for _, r in ipairs(rows) do total = total + r[col] end
  return total
end

local by_item, items = group_by(db.orders, "item")
for _, item in ipairs(items) do
  print($"{item}: {#by_item[item]} orders, {sum(by_item[item], 'amount')} total")
end


And `JOIN` — the honest nested-loop version, O(n·m) and proud of it,
which for a notebook's worth of rows is also the correct engineering
call. Note the collision rule: both tables have an `id`, and the left
side wins.


In [ ]:
-- A nested-loop inner join: the honest O(n*m) version, which for a
-- notebook's worth of rows is also the right one.
function join(left, right, on)
  local out = {}
  for _, l in ipairs(left) do
    for _, r in ipairs(right) do
      if on(l, r) then
        local both = {}
        for k, v in pairs(r) do both[k] = v end
        for k, v in pairs(l) do both[k] = v end   -- left wins a collision (id here)
        out[#out + 1] = both
      end
    end
  end
  return out
end

local placed = join(db.orders, db.customers,
  function(o, c) return o.customer_id == c.id end)
show(limit(placed, 4), { "id", "name", "city", "item", "amount" })


## Now, actual SQL

Three stages, each a screenful: **tokenize** the text, **parse** the
tokens into a tree, **compile** the tree into closures. This is the same
shape every real query engine has, minus forty years of optimizer.

The tokenizer is five string patterns tried in order. Everything SQL
keeps case-insensitive is lowercased at the door, so the parser never
thinks about it again.


In [ ]:
local KEYWORDS = {}
for word in ("select from where and or not order by asc desc limit"):gmatch("%a+") do
  KEYWORDS[word] = true
end

function tokenize(sql)
  local toks, pos = {}, 1
  local function push(kind, text)
    toks[#toks + 1] = { kind = kind, text = text }
  end
  while pos <= #sql do
    local rest = sql:sub(pos)
    local space = rest:match("^%s+")
    local num   = rest:match("^%d+%.?%d*")
    local str   = rest:match("^'[^']*'")
    local word  = rest:match("^[%a_][%w_]*")
    local op    = rest:match("^[<>]=") or rest:match("^<>") or rest:match("^[=<>%(%),%*]")
    if space then
      pos = pos + #space
    elseif num then
      push("number", num);  pos = pos + #num
    elseif str then
      push("string", str);  pos = pos + #str
    elseif word then
      local lower = word:lower()
      push(KEYWORDS[lower] and "keyword" or "name", KEYWORDS[lower] and lower or word)
      pos = pos + #word
    elseif op then
      push("op", op);       pos = pos + #op
    else
      error($"SQL: unexpected character {rest:sub(1, 1)::%q} at position {pos}", 0)
    end
  end
  return toks
end

for _, t in ipairs(tokenize("SELECT name FROM customers WHERE city = 'London'")) do
  io.write(t.kind, "(", t.text, ")  ")
end
print()


The parser is recursive descent, loosest binding first: `OR` under
`AND` under `NOT` under a comparison — each precedence level is a
tiny function that calls the next one down. `take` demands a token and
says what it wanted when it is disappointed, which is where good error
messages come from. Diluvium's `switch` handles the leaf tokens.


In [ ]:
function parse_select(sql)
  local toks = tokenize(sql)
  local at = 1
  local function peek() return toks[at] end
  local function take(kind, text)
    local t = toks[at]
    if not t or t.kind ~= kind or (text and t.text ~= text) then
      error($"SQL: expected {text or kind}, got {t and t.text or 'end of input'}", 0)
    end
    at = at + 1
    return t
  end
  local function accept(kind, text)
    local t = toks[at]
    if t and t.kind == kind and (not text or t.text == text) then
      at = at + 1
      return t
    end
  end

  -- Expressions, loosest first: OR binds less than AND, AND less than
  -- NOT, NOT less than a comparison. Each level is a tiny function.
  local parse_or
  local function parse_atom()
    if accept("op", "(") then
      local e = parse_or()
      take("op", ")")
      return e
    end
    if accept("keyword", "not") then return { op = "not", parse_atom() } end
    local t = toks[at]
    if not t then error("SQL: expected a value, got end of input", 0) end
    at = at + 1
    switch t.kind do
      case "number" then return { op = "value", value = tonumber(t.text) }
      case "string" then return { op = "value", value = t.text:sub(2, -2) }
      case "name"   then return { op = "column", name = t.text }
      default error($"SQL: expected a value, got {t.text}", 0)
    end
  end
  local function parse_cmp()
    local left = parse_atom()
    local op = accept("op", "=") or accept("op", "<>")
      or accept("op", "<=") or accept("op", ">=")
      or accept("op", "<") or accept("op", ">")
    if not op then return left end
    return { op = op.text, left, parse_atom() }
  end
  local function parse_and()
    local left = parse_cmp()
    while accept("keyword", "and") do left = { op = "and", left, parse_cmp() } end
    return left
  end
  parse_or = function()
    local left = parse_and()
    while accept("keyword", "or") do left = { op = "or", left, parse_and() } end
    return left
  end

  take("keyword", "select")
  local cols
  if accept("op", "*") then
    cols = "*"
  else
    cols = { take("name").text }
    while accept("op", ",") do cols[#cols + 1] = take("name").text end
  end
  take("keyword", "from")
  local q = { cols = cols, from = take("name").text }
  if accept("keyword", "where") then q.where = parse_or() end
  if accept("keyword", "order") then
    take("keyword", "by")
    q.order = take("name").text
    q.desc = accept("keyword", "desc") ~= nil
    if not q.desc then accept("keyword", "asc") end
  end
  if accept("keyword", "limit") then q.limit = tonumber(take("number").text) end
  if peek() then error($"SQL: unexpected {peek().text} after the query", 0) end
  return q
end

local q = parse_select("SELECT name, city FROM customers WHERE city = 'London' OR id > 4 ORDER BY name LIMIT 5")
print($"from {q.from}, {#q.cols} columns, order by {q.order}, limit {q.limit}")
print($"top of the where tree: {q.where.op}")


Compiling beats interpreting, even at toy scale: every node of the
`WHERE` tree becomes a closure over its children, so by the time rows
are tested there is no tree left — just nested function calls. This is
sixty lines because `switch` carries the operator table.


In [ ]:
-- Compile a WHERE tree into a plain predicate: every node becomes a
-- closure over its children. No interpreter loop is left by the time a
-- row is tested.
local function compile(expr)
  switch expr.op do
    case "value" then
      return function() return expr.value end
    case "column" then
      return function(row)
        if row[expr.name] == nil then error($"SQL: no column {expr.name}", 0) end
        return row[expr.name]
      end
    case "not" then
      local inner = compile(expr[1])
      return function(row) return not inner(row) end
    default
      local left, right = compile(expr[1]), compile(expr[2])
      switch expr.op do
        case "="  then return function(r) return left(r) == right(r) end
        case "<>" then return function(r) return left(r) ~= right(r) end
        case "<"  then return function(r) return left(r) <  right(r) end
        case ">"  then return function(r) return left(r) >  right(r) end
        case "<=" then return function(r) return left(r) <= right(r) end
        case ">=" then return function(r) return left(r) >= right(r) end
        case "and" then return function(r) return left(r) and right(r) end
        case "or"  then return function(r) return left(r) or  right(r) end
        default error($"SQL: cannot compile {tostring(expr.op)}", 0)
      end
  end
end

function run_sql(sql)
  local q = parse_select(sql)
  local rows = db[q.from] or error($"SQL: no table {q.from}", 0)
  if q.where then rows = where(rows, compile(q.where)) end
  -- Order before projection, so ORDER BY works on a column SELECT drops.
  if q.order then rows = order_by(rows, q.order, q.desc) end
  if q.limit then rows = limit(rows, q.limit) end
  if q.cols ~= "*" then rows = select_cols(rows, q.cols) end
  return rows
end

print("run_sql is ready")


And now the spelling works:


In [ ]:
show(run_sql("SELECT name, city FROM customers WHERE city = 'London'"))
print()
show(run_sql("SELECT item, amount FROM orders WHERE amount >= 90 ORDER BY amount DESC"))
print()
show(run_sql("SELECT id, item FROM orders WHERE item = 'valves' OR item = 'tape' LIMIT 4"))


A query language earns its keep by how it fails. Four broken queries,
four different messages — from the parser, the table lookup, and the
compiled column reference:


In [ ]:
-- A query language earns its keep by how it fails.
print(select(2, pcall(run_sql, "SELECT FROM customers")))
print(select(2, pcall(run_sql, "SELECT name FROM starships")))
print(select(2, pcall(run_sql, "SELECT name FROM customers WHERE age > 30")))
print(select(2, pcall(run_sql, "SELECT name FROM customers WHERE city = ")))


## A picture from a query

The engine's rows are ordinary Lua tables, so they flow straight into
the Lab's plotting. Revenue by city: join, group, sum, chart.


In [ ]:
local placed = join(db.orders, db.customers,
  function(o, c) return o.customer_id == c.id end)

local groups, cities = group_by(placed, "city")
local totals = {}
for i, city in ipairs(cities) do totals[i] = sum(groups[city], "amount") end

plot.bar(cities, totals, { title = "revenue by city" })


## What real SQLite would take — and what arrived instead

The recipe below is for a **guest-side** binding: SQLite compiled into the
runtime, callable as a Lua library from inside the sandbox. That has not
happened, and the recipe is unchanged:

- compile the `sqlite3.c` amalgamation into the WASI target beside the
  interpreter, with an `lsqlite3`-shaped Lua binding;
- `:memory:` databases only, at first — an in-memory database needs no
  filesystem, so the WASI build's lack of one costs nothing;
- accept the size: roughly another megabyte of WASM, which is why it should
  probably be a separate artifact rather than a tax on every page load.

What *did* arrive is the other half, and it is arguably the more useful
one: **the host has SQLite, and a program asks it for rows.** That is the
production shape too — `host/dhost_sql.c` answers the same two calls over
the system SQLite — so a guest written against it moves to the C host
unchanged.

The difference is a capability boundary rather than a library:

| | guest-side binding | host connector *(this is what exists)* |
| :--- | :--- | :--- |
| reached by | `require "sqlite"` | `host:sql/query`, `host:sql/exec` |
| who can use it | anything in the sandbox | only a program granted it |
| what it costs | +1 MB in every kernel | nothing until wired |
| in production | would need building | already how `dhost_sql.c` works |

The cell below runs the notebook's opening query — the same `SELECT` — the
hostcall way.


### The same query, asked of a host

A hostcall is a message on a queue, not a function call: `{tok, call, args}`
out on `host/calls`, `{tok, status, value}` back on `host/replies`, matched
by a **correlation token** because replies arrive in whatever order the host
answers them.

Since v5.5.1_build7 the runtime writes that loop for you. `host` is a guest
library — a permanent, baked in beside `queue` — so a program says what it
wants and not how to ask. `doc/Hostcall.md` is the protocol underneath, and
**SQLite, through a hostcall** shows both spellings side by side along with
the whole connector.

This needs a swarm-capable runtime with that library. It prints a skip line
rather than failing if yours is older.

In [ ]:
if type(swarm) ~= "table" or type(host) ~= "table" then
  print("SKIP -- needs the Lab swarm runner and the host library (v5.5.1_build7+)")
else
  swarm.stop()
  swarm.start{
    root = [[
      local out = queue.lookup("outbox")

      -- The deployment granted a directory; this names the database inside
      -- it. Nothing is opened until the first call is made.
      local db = host.sql.open("orders.db")

      db.exec("CREATE TABLE orders (item TEXT, amount INTEGER)")
      for _, row in ipairs({ { "widget", 120 }, { "gasket", 90 }, { "shim", 40 } }) do
        db.exec("INSERT INTO orders (item, amount) VALUES (?, ?)", row[1], row[2])
      end

      -- The notebook's opening query, run by SQLite rather than by the
      -- parser above. ORDER BY, which the Lua engine had to grow by hand.
      local got = db.query(
        "SELECT item, amount FROM orders WHERE amount >= ? ORDER BY amount DESC", 90)
      queue.push(out, ("%d row(s)"):format(#got.rows))
      for _, r in ipairs(got.rows) do
        queue.push(out, ("  %s  %s"):format(tostring(r[1]), tostring(r[2])))
      end

      -- And one the connector refuses. Transactions are host state held
      -- against a guest, and the v1 encoding has nowhere to put a handle
      -- that spans two calls -- so it is a refusal, not a surprise.
      --
      -- `try_exec` returns rather than raising, which is what a line that
      -- expects to be refused wants: value, status, detail.
      local _, status, detail = db.try_exec("BEGIN")
      queue.push(out, "BEGIN -> refused as designed (" .. tostring(status) .. "): "
        .. tostring(detail))
    ]],
    caps = { "queue:*", "host:sql/query", "host:sql/exec" },
    budget = { instructions = 5000000, memory_kb = 256 },
    max_instances = 4,
    connectors = { sql = { scope = "lab", access = "readwrite", max_result_rows = 128 } },
  }
  swarm.step(60)
  for _, line in ipairs(swarm.drain("root", "outbox")) do print(tostring(line)) end
  swarm.stop()
end